In [1]:
import json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [3]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [4]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
result = grade_by_model(dataset, "")

In [5]:
result

{'error': 'Missing required inputs',
 'message': "Unable to provide evaluation - both 'task' and 'solution' parameters are required",
 'strengths': [],
 'weaknesses': ['No task description provided',
  'No solution code provided for review'],
 'reasoning': 'The review template was provided but the actual task and solution content were not populated. Please provide the specific task requirements and AI-generated solution code to evaluate.',
 'score': 0,
 'instructions': 'Please call this function with:\n- task: A clear description of the coding task/requirements\n- solution: The AI-generated code solution to review'}